# Train on a Colab GPU

Runs `train.train` on a Colab GPU (docs/training.md §8). Everything else (data, evaluation, the
app) stays on the laptop.

**Before you start**

1. On the laptop: `docker compose run --rm app python -m data.manifest`, then
   `docker compose run --rm app python -m data.pack`, and upload `runs/packs/data-<hash>.zip` to
   `MyDrive/battlemap-walls/packs/` on Google Drive.
2. Push the commit you want to train.
3. Runtime → Change runtime type → a **GPU**.
4. If the repository is private, add a GitHub token as the Colab secret `GITHUB_TOKEN`
   (key icon on the left; read access to the repository is enough).

Checkpoints, `metrics.csv`, prediction images and MLflow logs are written straight to
`MyDrive/battlemap-walls/runs/`, so a disconnect loses at most one epoch: run the notebook
again and it resumes from `last.pt`. Afterwards, copy `runs/<run_name>/` and `runs/mlruns/` into
the laptop's `runs/` folder.

In [ ]:
# Settings: change these per run.
REPO = "https://github.com/Darkanath/huji-ai-engineering-battlemap-walls.git"
COMMIT = "main"  # branch, tag or commit hash to train; write the hash down for the report
CONFIG = "train/configs/stage1_synthetic.toml"
PACK = ""  # file name in MyDrive/battlemap-walls/packs/; "" = the newest pack

DRIVE = "/content/drive/MyDrive/battlemap-walls"
RUNS = f"{DRIVE}/runs"
PROCESSED = "/content/data/processed"  # local disk: much faster than reading from Drive
CODE = "/content/battlemap-walls"

## 1. Drive, GPU and code

In [ ]:
import os
import subprocess

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE}/packs", exist_ok=True)
os.makedirs(RUNS, exist_ok=True)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# Clone the repository at COMMIT (with the GITHUB_TOKEN secret if there is one).
try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = REPO.replace("https://", f"https://{token}@") if token else REPO
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--quiet", url, CODE], check=True)
subprocess.run(["git", "-C", CODE, "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "--quiet", COMMIT], check=True)
if COMMIT == "main":
    subprocess.run(["git", "-C", CODE, "pull", "--quiet", "origin", "main"], check=True)
print(
    subprocess.run(
        ["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True
    ).stdout
)

In [ ]:
# Install the project on Colab's own CUDA PyTorch: never reinstall torch here.
%pip install --quiet segmentation-models-pytorch albumentations mlflow
%pip install --quiet --no-deps -e {CODE}
os.chdir(CODE)
# Pretrained encoder weights are cached on Drive, so they download only once.
os.environ["HF_HOME"] = f"{DRIVE}/cache/huggingface"
os.environ["TORCH_HOME"] = f"{DRIVE}/cache/torch"

In [ ]:
import torch

print(torch.__version__, "CUDA:", torch.cuda.is_available())

## 2. Data pack → local disk

In [ ]:
import glob
import zipfile

packs = sorted(glob.glob(f"{DRIVE}/packs/data-*.zip"), key=os.path.getmtime)
pack = f"{DRIVE}/packs/{PACK}" if PACK else packs[-1]
os.makedirs(PROCESSED, exist_ok=True)
with zipfile.ZipFile(pack) as zf:
    zf.extractall(PROCESSED)
print(f"{pack}: {len(os.listdir(PROCESSED))} entries in {PROCESSED}")

## 3. Checks before the first full run (training.md §10)

Run these once per new setup, not before every run. The benchmark times 50 training steps and
one validation pass and estimates a full run from them, so you can tell whether a run fits one
Colab session (if not, it resumes in the next one). The overfit test trains on one batch for 200 steps: the loss must go close to 0,
otherwise something is wrong in the data, the loss or the model.

In [ ]:
import time
from pathlib import Path

from torch.utils.data import DataLoader

from data.manifest import read_manifest
from train.dataset import CropDataset, load_maps, select_rows
from train.losses import SegLoss, class_weights, split_shares
from train.model import autocast_dtype, build_model, pick_device
from train.train import load_config, make_optimizer, train_step, validate

cfg = load_config(Path(CONFIG))
data_cfg, t = cfg["data"], cfg["train"]
device = pick_device()
amp = autocast_dtype(device, cfg["runtime"]["precision"])
scaler = torch.amp.GradScaler("cuda") if amp == torch.float16 else None
rows = read_manifest(Path(PROCESSED) / "manifest.csv")
train_rows = select_rows(rows, "train", data_cfg["train"])
val_rows = select_rows(rows, "val", data_cfg["val"])
loss_fn = SegLoss(class_weights(split_shares(train_rows))).to(device)

# Benchmark: 50 training steps, then one validation pass over every val map.
steps = 50
crops = CropDataset(
    load_maps(Path(PROCESSED), train_rows, 256),
    steps * t["batch_size"],
    bias=data_cfg["crop_bias"],
    seed=cfg["seed"],
)
loader = DataLoader(crops, batch_size=t["batch_size"], num_workers=cfg["runtime"]["workers"])
model = build_model(encoder_weights="imagenet").to(device).train()
optimizer = make_optimizer(model, t)
started = time.time()
for images, masks in loader:
    train_step(model, images, masks, loss_fn, optimizer, device, amp, scaler)
if device.type == "cuda":
    torch.cuda.synchronize()
train_seconds = time.time() - started
val_data = load_maps(Path(PROCESSED), val_rows)
val_maps = [(r, m.image, m.mask) for r, m in zip(val_rows, val_data, strict=True)]
started = time.time()
validate(model, val_maps, loss_fn, device, cfg["val"])
val_seconds = time.time() - started
epoch = train_seconds * data_cfg["crops_per_epoch"] / (steps * t["batch_size"]) + val_seconds
print(
    f"{steps} steps: {train_seconds:.1f} s; validation ({len(val_maps)} maps): {val_seconds:.1f} s"
)
hours = epoch * t["epochs"] / 3600
print(f"full epoch about {epoch / 60:.1f} min; {t['epochs']} epochs about {hours:.1f} h")

In [ ]:
# Overfit test: one batch of 8 crops, 200 steps; the loss must fall close to 0.
crops = CropDataset(load_maps(Path(PROCESSED), train_rows[:8], 256), crops_per_epoch=8, seed=0)
batch = [crops[i] for i in range(8)]
images = torch.stack([image for image, _ in batch])
masks = torch.stack([mask for _, mask in batch])
model = build_model(encoder_weights="imagenet").to(device).train()
optimizer = make_optimizer(model, {**t, "lr_encoder": t["lr_decoder"]})
for step in range(200):
    loss = train_step(model, images, masks, loss_fn, optimizer, device, amp, scaler)
    if step % 40 == 0 or step == 199:
        print(f"step {step:3d}  loss {loss:.4f}")

## 4. Train

Resumes automatically if this run already has a `last.pt` on Drive. To start a run over, delete
its folder in `MyDrive/battlemap-walls/runs/` first.

In [ ]:
resume = Path(RUNS, cfg["run_name"], "last.pt").exists()
print("resuming" if resume else "new run", cfg["run_name"])
args = [
    "python",
    "-m",
    "train.train",
    "--config",
    CONFIG,
    "--runs",
    RUNS,
    "--processed",
    PROCESSED,
] + (["--resume"] if resume else [])
!{" ".join(args)}

## 5. Results

The per-epoch metrics of the run. Copy `MyDrive/battlemap-walls/runs/<run_name>/` and
`runs/mlruns/` to the laptop to look at them in the MLflow UI and to evaluate the checkpoint.

In [ ]:
import csv

with open(Path(RUNS, cfg["run_name"], "metrics.csv"), newline="") as f:
    for row in csv.DictReader(f):
        print(
            f"epoch {row['epoch']:>2}  train loss {float(row['train_loss']):.4f}  "
            f"real val wall BF1 {row.get('select/boundary_f1_wall', '-')}"
        )